# 19_final_training_data — 최종 학습데이터 구축 (1:1)

**이 노트북이 하는 일:** WEKA로 골라낸 descriptor + fingerprint + 정답(potency) + 분자 식별자(SMILES)를
하나의 표로 합쳐 **모델 학습에 바로 쓸 최종 데이터**를 만든다.

**구성:** `canonical_smiles`(식별자) + `ECFP4 fingerprint 1024개`(구조 지문) + `WEKA 선택 descriptor`(물성) + `potency`(정답)
**저장:** CSV와 Excel 두 형식

fingerprint(구조)와 descriptor(물성)를 함께 넣어, 두 표현의 정보를 모두 모델에 제공하는 것이 목적이다.

### 셀 1 — 준비 + fingerprint 생성기
루트 이동 후 라이브러리를 불러온다. `rdFingerprintGenerator.GetMorganGenerator`는
**ECFP4(Morgan 반경 2) fingerprint** 생성기로, 한 번 만들어 재사용하면 빠르다. 비트 수는 1024로 고정.

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator          # 최신 fingerprint API
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

FP_BITS = 1024                                          # fingerprint 비트 수
gen_ecfp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=FP_BITS)  # ECFP4 생성기

### 셀 2 — 입력 로드 (전체 descriptor Excel + WEKA 선택 목록)
두 파일을 읽는다: (1) 셀에서 만든 **전체 descriptor Excel**(canonical_smiles·potency·217 descriptor 포함),
(2) **WEKA가 고른 descriptor만 남은 CSV**의 헤더(=선택된 descriptor 이름 목록).
선택된 이름들이 전체 Excel 열에 다 존재하는지 `assert`로 확인한다(이름 불일치 방지).

In [ ]:
# 입력: (1) 전체 descriptor Excel(canonical_smiles+potency+217 desc)
#       (2) WEKA로 고른 descriptor 목록(filtered.csv 헤더)
FULL = 'data/HSD17B13_1to1_descriptors.xlsx'
FILT = 'data/HSD17B13_1to1_descriptors_weka_filtered.csv'

full = pd.read_excel(FULL)                                          # 전체(메타+217 descriptor)
sel_cols = [c for c in pd.read_csv(FILT, nrows=0).columns if c.lower() != 'potency']  # 선택된 descriptor 이름들
print('WEKA 선택 descriptor:', len(sel_cols), '개')

miss = [c for c in sel_cols if c not in full.columns]              # 전체에 없는 이름이 있으면 오류
assert not miss, ('full에 없는 descriptor: %s' % miss)
print('전체 화합물:', len(full), '| potency 분포:', dict(full.potency.value_counts()))

### 셀 3 — canonical SMILES → ECFP4 fingerprint(1024비트)
각 분자를 fingerprint(0/1 비트 1024개)로 변환한다. `GetFingerprintAsNumPy`가 numpy 배열로 바로 준다.
파싱 실패한 분자는 건너뛰고(keep에 성공 행만 기록), 나중에 descriptor·potency와 행을 맞춘다.
열 이름은 `fp_0000` ~ `fp_1023`.

In [ ]:
# canonical SMILES -> ECFP4 fingerprint(1024 bit)
fp_rows, keep = [], []
for i, smi in enumerate(full['canonical_smiles']):
    m = Chem.MolFromSmiles(str(smi))
    if m is None:                                        # 파싱 실패 분자는 제외
        continue
    fp_rows.append(gen_ecfp.GetFingerprintAsNumPy(m))    # 1024비트 배열
    keep.append(i)
FP = pd.DataFrame(np.vstack(fp_rows).astype(np.int8),    # 비트라 int8로 저장(용량↓)
                  columns=['fp_%04d' % j for j in range(FP_BITS)])
base = full.iloc[keep].reset_index(drop=True)            # fingerprint 성공한 행만 남긴 메타+descriptor
print('fingerprint 계산 완료:', FP.shape, '| 유효 화합물', len(base))

### 셀 4 — 최종 조립 & 저장 (CSV + Excel)
네 조각을 좌우로 이어 붙인다: **식별자(SMILES) → fingerprint(1024) → 선택 descriptor → potency(맨 끝)**.
potency를 마지막 열에 두는 건 ML/WEKA에서 클래스 열을 끝에 두는 관례를 따른 것.
같은 표를 CSV와 Excel로 각각 저장한다.

In [ ]:
# 최종 조립: canonical_smiles + fingerprint(1024) + descriptor(선택) + potency
final = pd.concat([
    base[['canonical_smiles']].reset_index(drop=True),   # 1) 식별자
    FP.reset_index(drop=True),                           # 2) fingerprint 1024
    base[sel_cols].reset_index(drop=True),               # 3) WEKA 선택 descriptor
    base[['potency']].reset_index(drop=True),            # 4) 클래스(마지막 열)
], axis=1)
print('최종 학습데이터 shape:', final.shape,
      '(= canonical_smiles 1 + fp %d + desc %d + potency 1)' % (FP_BITS, len(sel_cols)))
print('구성 확인 → 첫 열:', final.columns[0], '| 마지막 열:', final.columns[-1])

OUT_CSV = 'data/HSD17B13_final_training_1to1.csv'
OUT_XLSX = 'data/HSD17B13_final_training_1to1.xlsx'
final.to_csv(OUT_CSV, index=False)                       # CSV 저장
final.to_excel(OUT_XLSX, index=False)                    # Excel 저장
print('저장 완료:')
print('  CSV :', OUT_CSV)
print('  XLSX:', OUT_XLSX)